# LLaMA 2 7B + QLoRA - Semantic Text Classification

Setup:
1. Runtime > Change runtime type > T4 GPU
2. Upload `dmdw_llama2.zip` via Files sidebar (left, folder icon)
3. Run cells in order

In [ ]:
!nvidia-smi

In [ ]:
import os, glob, zipfile, shutil

PROJECT = 'Data-mining-semantic-w-LlaMa2'
os.chdir('/content')

if not os.path.isdir(PROJECT):
    zips = sorted(glob.glob('/content/*.zip'))
    assert zips, 'Upload dmdw_llama2.zip first (Files sidebar > drag-drop)'
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall('/content')
    print(f'extracted: {zips[0]}')

os.chdir(f'/content/{PROJECT}')
print('cwd:', os.getcwd())
print('files:', sorted(os.listdir()))

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
try:
    from google.colab import userdata
    for key in ['HF_TOKEN', 'WANDB_API_KEY']:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass
except ImportError:
    pass

from src.utils import Env
logged = Env().load().hf_login()
assert logged, 'HF_TOKEN missing - add Colab Secret or edit .env'

In [ ]:
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    target = os.environ.get('DRIVE_MOUNT', '/content/drive/MyDrive/dmdw_llama2')
    os.makedirs(target, exist_ok=True)
    if os.path.islink('results') or os.path.exists('results'):
        if os.path.islink('results'):
            os.unlink('results')
        else:
            shutil.rmtree('results')
    os.symlink(target, 'results')
    print('results ->', target)

In [ ]:
!python main.py --dataset ag_news --runs 1 --smoke --no-hp --no-ablation --no-scale

In [ ]:
!python main.py --dataset ag_news --runs 1 --no-hp --no-ablation --no-scale

In [ ]:
!python main.py --dataset dbpedia_14 --runs 1 --no-hp --no-ablation --no-scale

In [ ]:
!python main.py --dataset both

In [ ]:
import json
for ds in ['ag_news', 'dbpedia_14']:
    path = f'results/{ds}/all_results.json'
    if os.path.exists(path):
        with open(path) as fh:
            data = json.load(fh)
        print(f'\n=== {ds} ===')
        for k, v in data.get('metrics_summary', {}).items():
            if isinstance(v, dict) and 'mean' in v:
                print(f'  {k:30s}: {v["mean"]:.4f} +/- {v["std"]:.4f}')